<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

I checked impressions and clicks. Both have heavy tails. This means a tiny number of pages get almost all the traffic, while the vast majority get zero or close to zero.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect DuckDB to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection verified and ready.")

Warehouse connection verified and ready.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check distributions using quantiles
q_dist = f"""
SELECT
    quantile_cont(gsc_impressions, 0.5) AS p50_impressions,
    quantile_cont(gsc_impressions, 0.9) AS p90_impressions,
    quantile_cont(gsc_impressions, 0.99) AS p99_impressions,
    quantile_cont(gsc_clicks, 0.5) AS p50_clicks,
    quantile_cont(gsc_clicks, 0.9) AS p90_clicks,
    quantile_cont(gsc_clicks, 0.99) AS p99_clicks
FROM {table_path}
WHERE month = '2026-03'
"""
print("Distributions (Heavy Tails Check):")
print(con.sql(q_dist).df().to_string(index=False))

Distributions (Heavy Tails Check):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 p50_impressions  p90_impressions  p99_impressions  p50_clicks  p90_clicks  p99_clicks
             0.0             54.0            509.0         0.0         0.0         2.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*


Signal 1: More impressions lead to more clicks. Verdict: CONFIRMED.

Signal 2: Better ranking position means higher CTR. Verdict: CONFIRMED.

Signal 3: Higher scroll events mean higher engaged sessions. Verdict: CONFIRMED.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Signal 1: Impressions vs Clicks")
print(con.sql(f"""
    SELECT
        CASE WHEN gsc_impressions > 1000 THEN 'High' ELSE 'Low' END as vol,
        AVG(gsc_clicks) as avg_clicks
    FROM {table_path} WHERE month = '2026-03' GROUP BY 1
""").df())

print("\nSignal 2: Position vs CTR")
print(con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position <= 10 THEN 'Page 1' ELSE 'Page 2+' END as pos,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM {table_path} WHERE month = '2026-03' AND gsc_impressions > 10 GROUP BY 1
""").df())

print("\nSignal 3: Scrolls vs Engaged Sessions")
print(con.sql(f"""
    SELECT
        CASE WHEN scroll_events > 0 THEN 'Has Scrolls' ELSE 'No Scrolls' END as scroll_status,
        AVG(ga4_engaged_sessions) as avg_engaged
    FROM {table_path} WHERE month = '2026-03' GROUP BY 1
""").df())

Signal 1: Impressions vs Clicks
    vol  avg_clicks
0   Low    0.067045
1  High    5.073857

Signal 2: Position vs CTR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       pos   avg_ctr
0   Page 1  0.003356
1  Page 2+  0.001903

Signal 3: Scrolls vs Engaged Sessions


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  scroll_status  avg_engaged
0    No Scrolls     0.000121
1   Has Scrolls     0.233574


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank flags pages on Page 1 that get zero clicks. I tested the assumption that Page 1 positions act equally. Verdict: FALSE. The CTR drop-off on Page 1 is massive. A page at position 2 gets clicks, but a page at position 9 naturally gets almost zero, so flagging position 9 as an anomaly is wrong.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Flag-Linked Test: Page 1 Position Drop-off")
print(con.sql(f"""
    SELECT
        FLOOR(gsc_avg_position) AS exact_position,
        COUNT(*) as pages,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM {table_path}
    WHERE month = '2026-03'
      AND gsc_avg_position >= 1
      AND gsc_avg_position <= 10
      AND gsc_impressions > 50
    GROUP BY 1
    ORDER BY 1
""").df())

Flag-Linked Test: Page 1 Position Drop-off
   exact_position   pages   avg_ctr
0             1.0   65952  0.003909
1             2.0  102768  0.004117
2             3.0  114508  0.003800
3             4.0  107941  0.003407
4             5.0   94326  0.003184
5             6.0   71925  0.003147
6             7.0   50978  0.003249
7             8.0   36418  0.002827
8             9.0   26630  0.003144
9            10.0     223  0.002868


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The data shows top positions take almost all the traffic. A content team should focus their refresh efforts on pages ranking just outside the top 3 (positions 4-6), as moving up a few spots yields huge returns. Pages ranking at the bottom of page 1 naturally get low clicks and shouldn't trigger false alarms.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.